In [2]:
import io
import requests
from bs4 import BeautifulSoup
import pandas as pd
import yfinance as yf

# Question 3. [Index] S&P 500 Market Corrections Analysis
# Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.


# Download S&P 500 historical data
sp500 = yf.download('^GSPC', start='1950-01-01', progress=False)
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

close = sp500['Close'].dropna()
cummax = close.cummax()
ath_days = close[close == cummax]
ath_dates = ath_days.index

records_closed = []
for i in range(len(ath_dates) - 1):
    start_dt = ath_dates[i]
    end_dt = ath_dates[i+1]
    
    sub = close.loc[(close.index >= start_dt) & (close.index < end_dt)]
    min_val = sub.min()
    min_dt = sub.idxmin()
    high = close.loc[start_dt]
    
    dd = (high - min_val) / high * 100
    duration = (min_dt - start_dt).days
    records_closed.append({
        'start_dt': start_dt,
        'end_dt': end_dt,
        'trough_dt': min_dt,
        'high': high,
        'low': min_val,
        'drawdown': dd,
        'duration_days': duration
    })

df_closed = pd.DataFrame(records_closed)
df_corr_closed = df_closed[df_closed['drawdown'] >= 5].copy()

print(f"Total corrections >= 5% (closed pairs): {len(df_corr_closed)}")
print("Percentiles for closed pairs:")
print("Drawdown (%):")
print(f"25th: {df_corr_closed['drawdown'].quantile(0.25):.2f}%")
print(f"50th (median): {df_corr_closed['drawdown'].quantile(0.50):.2f}%")
print(f"75th: {df_corr_closed['drawdown'].quantile(0.75):.2f}%")
print("Duration (days):")
print(f"25th: {df_corr_closed['duration_days'].quantile(0.25):.2f} days")
print(f"50th (median): {df_corr_closed['duration_days'].quantile(0.50):.2f} days")
print(f"75th: {df_corr_closed['duration_days'].quantile(0.75):.2f} days")

# Now check if we include the last ATH to present (if last price < cummax.iloc[-1])
records_with_ongoing = list(records_closed)
last_ath_dt = ath_dates[-1]
if last_ath_dt < close.index[-1]:
    sub = close.loc[close.index >= last_ath_dt]
    min_val = sub.min()
    min_dt = sub.idxmin()
    high = close.loc[last_ath_dt]
    dd = (high - min_val) / high * 100
    duration = (min_dt - last_ath_dt).days
    records_with_ongoing.append({
        'start_dt': last_ath_dt,
        'end_dt': close.index[-1],
        'trough_dt': min_dt,
        'high': high,
        'low': min_val,
        'drawdown': dd,
        'duration_days': duration
    })

df_ongoing = pd.DataFrame(records_with_ongoing)
df_corr_ongoing = df_ongoing[df_ongoing['drawdown'] >= 5].copy()
print(f"\nTotal corrections >= 5% (with ongoing): {len(df_corr_ongoing)}")
print("Percentiles with ongoing:")
print("Drawdown (%):")
print(f"25th: {df_corr_ongoing['drawdown'].quantile(0.25):.2f}%")
print(f"50th (median): {df_corr_ongoing['drawdown'].quantile(0.50):.2f}%")
print(f"75th: {df_corr_ongoing['drawdown'].quantile(0.75):.2f}%")
print("Duration (days):")
print(f"25th: {df_corr_ongoing['duration_days'].quantile(0.25):.2f} days")
print(f"50th (median): {df_corr_ongoing['duration_days'].quantile(0.50):.2f} days")
print(f"75th: {df_corr_ongoing['duration_days'].quantile(0.75):.2f} days")

# Check trading days vs calendar days
df_corr_closed['duration_trading_days'] = [
    len(close.loc[(close.index >= r['start_dt']) & (close.index <= r['trough_dt'])]) - 1
    for _, r in df_corr_closed.iterrows()
]
print("\nTrading days duration percentiles:")
print(f"25th: {df_corr_closed['duration_trading_days'].quantile(0.25):.2f}")
print(f"50th: {df_corr_closed['duration_trading_days'].quantile(0.50):.2f}")
print(f"75th: {df_corr_closed['duration_trading_days'].quantile(0.75):.2f}")

# === FINAL ANSWER ===
median_drawdown = df_corr_closed['drawdown'].quantile(0.50)
median_duration = df_corr_closed['duration_days'].quantile(0.50)

print("\n" + "="*50)
print("QUESTION 3 — FINAL ANSWER")
print("="*50)
print(f"  Significant corrections (>= 5% drawdown): {len(df_corr_closed)}")
print(f"  Median Drawdown  : {median_drawdown:.2f}%  --> Rounded Answer: {round(median_drawdown)}")
print(f"  Median Duration  : {median_duration:.1f} days")
print("="*50)



Total corrections >= 5% (closed pairs): 74
Percentiles for closed pairs:
Drawdown (%):
25th: 6.23%
50th (median): 7.99%
75th: 14.02%
Duration (days):
25th: 22.00 days
50th (median): 40.50 days
75th: 86.25 days

Total corrections >= 5% (with ongoing): 74
Percentiles with ongoing:
Drawdown (%):
25th: 6.23%
50th (median): 7.99%
75th: 14.02%
Duration (days):
25th: 22.00 days
50th (median): 40.50 days
75th: 86.25 days

Trading days duration percentiles:
25th: 14.25
50th: 27.50
75th: 61.25

QUESTION 3 — FINAL ANSWER
  Significant corrections (>= 5% drawdown): 74
  Median Drawdown  : 7.99%  --> Rounded Answer: 8
  Median Duration  : 40.5 days
